In [1]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START, END
from typing import Literal,TypedDict
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [2]:
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.7)

In [3]:
generator_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")


In [4]:
class TweetState(TypedDict):
      topic: str
      tweet: str
      evaluation: Literal["approved", "need improvement"]
      feedback: str

      iteration: int = 0
      max_iterations: int

In [5]:
def generate_tweet(state: TweetState) -> TweetState:
    prompt = f"Generate a tweet about {state['topic']} in less than 280 characters."
    tweet = generator_llm(prompt)
    return {"tweet": tweet, "evaluation": "", "feedback": "", "iteration": 0, "max_iterations": 3}

In [6]:
def evaluate_tweet(state: TweetState) -> TweetState:
    prompt = f"Evaluate the following tweet about {state['topic']} and provide feedback for improvement: {state['tweet']}"
    evaluation = evaluator_llm(prompt)
    return {"evaluation": "approved" if "good" in evaluation else "need improvement", "feedback": evaluation, "iteration": state["iteration"], "max_iterations": state["max_iterations"]}

In [7]:
def optimize_tweet(state: TweetState) -> TweetState:
    prompt = f"Optimize the following tweet based on the feedback: {state['tweet']} Feedback: {state['feedback']}"
    optimized_tweet = optimizer_llm(prompt)
    return {"tweet": optimized_tweet, "evaluation": "", "feedback": "", "iteration": state["iteration"] + 1, "max_iterations": state["max_iterations"]}

In [8]:
graph = StateGraph(TweetState)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)